# MTI830 — Pipeline de sélection des joueurs et des parties

Ce notebook exécute la pipeline suivante :

1. Charger une liste manuelle de joueurs candidats depuis `players.txt`
2. Télécharger leurs parties brutes via l'API Lichess/Berserk
3. Sauvegarder le brut API (`raw_games_by_player.json`)
4. Construire `player_games_raw` via `build_player_games(...)`
5. Sauvegarder `player_games_raw.json`
6. Appliquer `select_player_games(...)`
7. Inspecter les résultats

Avant d'exécuter le notebook, vérifie que :

- `players.txt` existe à la racine du projet
- tes modules `src.ingestion.player_sampling`, `src.dataset.io`, `src.dataset.select_players` sont à jour
- ton token API Lichess est disponible


## 0. Imports et configuration


In [18]:
from pathlib import Path
import os
import sys
sys.path.append(os.path.abspath(".."))
import json
from dotenv import load_dotenv
import time
import datetime 
from datetime import timedelta

import pandas as pd
import berserk

from src.ingestion.player_sampling import collect_raw_games_for_players
from src.dataset.io import (
    save_raw_games_by_player,
    load_raw_games_by_player,
    build_player_games_raw_from_api_dump,
    save_player_games_raw,
    load_player_games_raw,
    save_raw_games_for_one_player,
    list_downloaded_player_ids,
    load_raw_games_by_player_from_directory,
    save_player_games_selected
)
from src.dataset.select_players import PlayerSelectionConfig, select_player_games


In [16]:
# Ajuste ce chemin si nécessaire
PLAYERS_TXT_PATH = Path("../data/player_candidates/players.txt")
RAW_GAMES_JSON_PATH = Path("../data/raw/raw_games_by_player.json")
PLAYER_GAMES_RAW_JSON_PATH = Path("../data/raw/player_games_raw.json")
RAW_GAMES_BY_PLAYER_DIR = Path("../data/raw/by_player")
FAILED_PLAYERS_PATH = Path("../data/raw/failed_players.txt")
PLAYER_GAMES_SELECTED_JSON_PATH = Path("../data/selected/player_games_selected.json")
SELECTED_PLAYERS_TXT_PATH = Path("../data/selected/selected_players.txt")
SELECTION_SUMMARY_JSON_PATH = Path("../data/selected/selection_summary.json")

# Pour un premier test, garde une limite raisonnable
MAX_GAMES_PER_PLAYER = 1100
SLEEP_SECONDS = 0.3

# Option pratique : recharger depuis disque au lieu de retélécharger
USE_EXISTING_RAW_GAMES_JSON = False
USE_EXISTING_PLAYER_GAMES_RAW_JSON = False

START_2025 = berserk.utils.to_millis(datetime.datetime(2025, 1, 1))
END_2026 = berserk.utils.to_millis(datetime.datetime(2026, 1, 1))


## 1. Création du client Berserk


In [3]:
# Option 1 : variable d'environnement LICHESS_API_TOKEN
load_dotenv("../.env")
token = os.getenv('LICHESS_API_TOKEN')

# Option 2 : décommente et colle temporairement ton token si besoin
# token = 'COLLE_ICI_TON_TOKEN'

if not token:
    raise ValueError('Token API introuvable. Définis LICHESS_API_TOKEN ou renseigne token dans la cellule.')

session = berserk.TokenSession(token)
client = berserk.Client(session=session)

print('Client Berserk créé avec succès')


Client Berserk créé avec succès


## 2. Charger la liste manuelle de joueurs


In [4]:
def load_player_ids_from_txt(path: str | Path) -> list[str]:
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    return [line.strip() for line in lines if line.strip()]

seed_player_ids = load_player_ids_from_txt(PLAYERS_TXT_PATH)
print(f'{len(seed_player_ids)} joueurs chargés')
print(seed_player_ids[:15])

downloaded_player_ids = set(list_downloaded_player_ids(RAW_GAMES_BY_PLAYER_DIR))
remaining_player_ids = [p for p in seed_player_ids if p not in downloaded_player_ids]

print(f"Déjà téléchargés : {len(downloaded_player_ids)}")
print(f"Restants à télécharger : {len(remaining_player_ids)}")


890 joueurs chargés
['VW-Leoo', 'Lalsahab8855', 'ElizavetaVesna', 'nevo01', 'plotogamer', 'thelegendxd8bpq', 'MattMatze', 'ramya254', 'ajedrecista28', 'igraclagan', 'gr33do', 'sefeydn', 'MilanKontic', 'std_priority_queue', 'cool_shine']
Déjà téléchargés : 864
Restants à télécharger : 26


## 3. Télécharger ou recharger `raw_games_by_player`


In [5]:
downloaded_player_ids = set(list_downloaded_player_ids(RAW_GAMES_BY_PLAYER_DIR))
remaining_player_ids = [p for p in seed_player_ids if p not in downloaded_player_ids]

print(f"Déjà téléchargés : {len(downloaded_player_ids)}")
print(f"Restants à télécharger : {len(remaining_player_ids)}")
print(remaining_player_ids[:10])

Déjà téléchargés : 864
Restants à télécharger : 26
['vebbev', 'Velociraptor1510', 'starfox_88', 'Fnntnn', 'BENHAIMED-CMPT3', 'imbayog', 'Egordan', 'sunil_goyal', 'nowjohn', 'JohnPion']


In [6]:
failed_players = []

for idx, player_id in enumerate(remaining_player_ids, start=1):
    player_json_path = RAW_GAMES_BY_PLAYER_DIR / f"{player_id}.json"

    # Garde-fou de reprise
    if player_json_path.exists():
        print(f"[SKIP] {player_id} déjà téléchargé")
        continue

    try:
        # 1) Données publiques du joueur
        user_data = client.users.get_public_data(player_id)

        if not user_data or "createdAt" not in user_data:
            raise ValueError("Champ 'createdAt' introuvable dans get_public_data.")

        created_at = user_data["createdAt"]

        # 2) Fenêtre d'observation : 6 mois après création du compte
        since = berserk.utils.to_millis(created_at)
        until = berserk.utils.to_millis(created_at + timedelta(days=183))

        # 3) Export des parties dans cette fenêtre
        games = client.games.export_by_player(
            player_id,
            max=MAX_GAMES_PER_PLAYER,
            since=since,
            until=until,
            perf_type="rapid",
            opening=True,
        )
        games = list(games)

        # 4) Filtrage local complémentaire
        games = [
            g for g in games
            if isinstance(g, dict)
            and g.get("rated") is True
            and (
                str(g.get("speed") or "").lower() == "rapid"
                or str(g.get("perf") or "").lower() == "rapid"
            )
            and str(g.get("status") or "").lower() not in {"created", "started", ""}
        ]

        # 5) Tri du plus ancien au plus récent
        games = sorted(games, key=lambda g: g.get("createdAt"))

        # 6) Sauvegarde immédiate
        save_raw_games_for_one_player(
            player_id=player_id,
            games=games,
            directory=RAW_GAMES_BY_PLAYER_DIR,
        )

        print(
            f"[{idx}/{len(remaining_player_ids)}] "
            f"{player_id} | createdAt={created_at} | {len(games)} parties sauvegardées"
        )

    except Exception as e:
        failed_players.append(player_id)
        print(f"[ERREUR] {player_id}: {e}")

        with FAILED_PLAYERS_PATH.open("a", encoding="utf-8") as f:
            f.write(player_id + "\n")

        # si rate limit, pause plus longue
        if "429" in str(e) or "Too Many Requests" in str(e):
            print("Pause longue après rate limit...")
            time.sleep(30)

    time.sleep(SLEEP_SECONDS)

[ERREUR] vebbev: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Velociraptor1510: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] starfox_88: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Fnntnn: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] BENHAIMED-CMPT3: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] imbayog: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Egordan: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] sunil_goyal: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] nowjohn: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] JohnPion: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] D171016350: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] deVKim: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] unkownahirishere: Champ 'createdAt' introuvable dans get_public_data.
[ERREUR] Simo01101110: Champ 'createdAt' introuvable dan

In [7]:
raw_games_by_player = load_raw_games_by_player_from_directory(RAW_GAMES_BY_PLAYER_DIR)

print(f"raw_games_by_player reconstruit depuis {RAW_GAMES_BY_PLAYER_DIR}")
print(f"{len(raw_games_by_player)} joueurs chargés depuis le disque")

raw_games_by_player reconstruit depuis ..\data\raw\by_player
864 joueurs chargés depuis le disque


## 4. Construire ou recharger `player_games_raw`

Ici, on passe du JSON API brut à la table analytique.
La fonction `build_player_games_raw_from_api_dump(...)` doit :

- appeler `build_player_games(games)` pour chaque joueur
- concaténer les DataFrames
- faire `drop_duplicates(subset=['player_id', 'game_id'])`


In [8]:
if USE_EXISTING_PLAYER_GAMES_RAW_JSON and PLAYER_GAMES_RAW_JSON_PATH.exists():
    player_games_raw = load_player_games_raw(PLAYER_GAMES_RAW_JSON_PATH)
    print(f'player_games_raw rechargé depuis {PLAYER_GAMES_RAW_JSON_PATH}')
else:
    player_games_raw = build_player_games_raw_from_api_dump(raw_games_by_player)
    save_player_games_raw(player_games_raw, PLAYER_GAMES_RAW_JSON_PATH)
    print(f'player_games_raw sauvegardé dans {PLAYER_GAMES_RAW_JSON_PATH}')


player_games_raw sauvegardé dans ..\data\raw\player_games_raw.json


In [9]:
print(player_games_raw.shape)
display(player_games_raw.head())


(537308, 22)


,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,termination_type,has_increment,opening_family,opening_eco,opening_name,ply_count,perf,speed,rated,source
0,Oa0VdpEH,2023-09-27 01:43:00.646000+00:00,2,aadhavanpca,AadhavanPCA,dishikarajput,dishikarajput,white,1.0,800,...,mate,1,Queen's Pawn,A40,Mikenas Defense,73,rapid,rapid,True,swiss
1,Oa0VdpEH,2023-09-27 01:43:00.646000+00:00,2,dishikarajput,dishikarajput,aadhavanpca,AadhavanPCA,black,0.0,472,...,mate,1,Queen's Pawn,A40,Mikenas Defense,73,rapid,rapid,True,swiss
2,xuMep03q,2023-09-27 01:52:25.117000+00:00,2,jyoana,Jyoana,aadhavanpca,AadhavanPCA,white,0.0,842,...,mate,1,Alekhine Defense & related starts,B01,Scandinavian Defense: Mieses-Kotroc Variation,90,rapid,rapid,True,swiss
3,xuMep03q,2023-09-27 01:52:25.117000+00:00,2,aadhavanpca,AadhavanPCA,jyoana,Jyoana,black,1.0,800,...,mate,1,Alekhine Defense & related starts,B01,Scandinavian Defense: Mieses-Kotroc Variation,90,rapid,rapid,True,swiss
4,lphYNycn,2023-09-27 02:06:03.248000+00:00,2,aadhavanpca,AadhavanPCA,theivaan,theivaan,white,1.0,879,...,mate,1,Queen’s Pawn Systems,D00,Amazon Attack,73,rapid,rapid,True,swiss


In [10]:
print('Nb joueurs dans player_games_raw :', player_games_raw['player_id'].nunique())
print('Doublons (player_id, game_id) :', player_games_raw.duplicated(subset=['player_id', 'game_id']).sum())


Nb joueurs dans player_games_raw : 174967
Doublons (player_id, game_id) : 0


## 5. Appliquer la sélection finale


In [11]:
selection_config = PlayerSelectionConfig(
    min_games_in_window=200,
    max_games_in_window=1000,
    burnin_games=20,
    min_initial_elo=1000,
    max_initial_elo=1400,
    observation_window_days=183,
)

player_games_selected = select_player_games(
    player_games_raw,
    config=selection_config,
    verbose=True,
)


[Initial] rows=537,308 | players=174,967
[Après filtres bruts] rows=537,308 | players=174,967
[Après suppression des 20 premières parties rapid/rated/finished] rows=259,072 | players=611
[Après filtre Elo initial [1000, 1400]] rows=160,788 | players=388
[Après fenêtre d'observation de 183 jours] rows=160,371 | players=388
[Après filtre nombre de parties [200, 1000]] rows=70,620 | players=143

[Résumé final]
Joueurs retenus : 143
Lignes retenues  : 70,620
Nb de parties par joueur :
count    143.000000
mean     493.846154
std      211.356428
min      202.000000
25%      327.000000
50%      460.000000
75%      636.500000
max      998.000000
Name: game_id, dtype: float64

Elo initial dans la fenêtre :
count     143.000000
mean     1167.419580
std        97.754393
min      1007.000000
25%      1091.000000
50%      1165.000000
75%      1245.500000
max      1400.000000
Name: elo_player_before, dtype: float64


## 6. Inspecter les résultats


In [12]:
print('Nb joueurs retenus :', player_games_selected['player_id'].nunique())
print(player_games_selected.groupby('player_id')['game_id'].count().describe())
display(player_games_selected[['player_id', 'elo_player_before']].head())


Nb joueurs retenus : 143
count    143.000000
mean     493.846154
std      211.356428
min      202.000000
25%      327.000000
50%      460.000000
75%      636.500000
max      998.000000
Name: game_id, dtype: float64


,player_id,elo_player_before
0,abdeerrahim,1129
1,abdeerrahim,1143
2,abdeerrahim,1127
3,abdeerrahim,1140
4,abdeerrahim,1125


In [13]:
selected_players = sorted(player_games_selected['player_id'].unique())
print(len(selected_players))
print(selected_players[:30])


143
['abdeerrahim', 'ad0105', 'adalbertus_szachmat', 'ahmetsnll', 'albina228', 'alessandro949', 'alexanderororo', 'alexey963', 'alexsur99', 'alexviat', 'aluschirm', 'ameliajuniorchess', 'anje36', 'antoniobotafogo-1957', 'armansingh11', 'arsham105', 'artemasa', 'ashwinarokia', 'assassin72', 'aureliawlkn', 'ayse41', 'bahimmm0', 'bartekchessplayer123', 'beenlivinalie', 'belarbia', 'bendjedidi_aymen2', 'benladen22', 'bianou', 'bigminipawn', 'butkovich']


In [15]:
games_per_player = player_games_selected.groupby('player_id')['game_id'].count().sort_values(ascending=False)
display(games_per_player.head(143))


player_id
alessandro949      998
erik_osl           995
maxicuro           978
nikita_chel        964
hajimh             949
                  ... 
serhatt0           227
patr188            205
bianou             205
mrrsm              205
chessskullscott    202
Name: game_id, Length: 143, dtype: int64

## 7. Sauvegardes complémentaires optionnelles

Si tu veux plus tard, tu pourras ajouter ici :

- sauvegarde de `player_games_selected`
- export de la liste finale des joueurs retenus
- statistiques résumées


In [19]:

# 1. Sauvegarde du dataset final sélectionné
save_player_games_selected(
    player_games_selected,
    PLAYER_GAMES_SELECTED_JSON_PATH,
)
print(f"player_games_selected sauvegardé dans {PLAYER_GAMES_SELECTED_JSON_PATH}")

# 2. Sauvegarde de la liste des joueurs retenus
selected_players = sorted(player_games_selected["player_id"].unique())
SELECTED_PLAYERS_TXT_PATH.parent.mkdir(parents=True, exist_ok=True)
SELECTED_PLAYERS_TXT_PATH.write_text("\n".join(selected_players), encoding="utf-8")
print(f"Liste des joueurs retenus sauvegardée dans {SELECTED_PLAYERS_TXT_PATH}")

# 3. Sauvegarde d'un résumé de sélection
games_per_player = player_games_selected.groupby("player_id")["game_id"].count()
first_games = (
    player_games_selected
    .sort_values(["player_id", "datetime_utc", "game_id"])
    .groupby("player_id", as_index=False)
    .first()
)

selection_summary = {
    "n_players": int(player_games_selected["player_id"].nunique()),
    "n_rows": int(len(player_games_selected)),
    "games_per_player": {
        "count": float(games_per_player.count()),
        "mean": float(games_per_player.mean()),
        "std": float(games_per_player.std()),
        "min": float(games_per_player.min()),
        "25%": float(games_per_player.quantile(0.25)),
        "50%": float(games_per_player.quantile(0.50)),
        "75%": float(games_per_player.quantile(0.75)),
        "max": float(games_per_player.max()),
    },
    "initial_elo_in_window": {
        "count": float(first_games["elo_player_before"].count()),
        "mean": float(first_games["elo_player_before"].mean()),
        "std": float(first_games["elo_player_before"].std()),
        "min": float(first_games["elo_player_before"].min()),
        "25%": float(first_games["elo_player_before"].quantile(0.25)),
        "50%": float(first_games["elo_player_before"].quantile(0.50)),
        "75%": float(first_games["elo_player_before"].quantile(0.75)),
        "max": float(first_games["elo_player_before"].max()),
    },
}

SELECTION_SUMMARY_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with SELECTION_SUMMARY_JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(selection_summary, f, ensure_ascii=False, indent=2)

print(f"Résumé de sélection sauvegardé dans {SELECTION_SUMMARY_JSON_PATH}")

player_games_selected sauvegardé dans ..\data\selected\player_games_selected.json
Liste des joueurs retenus sauvegardée dans ..\data\selected\selected_players.txt
Résumé de sélection sauvegardé dans ..\data\selected\selection_summary.json
